# COVID-19 Global Data Analysis
**Author:** Sankalp Chudmunge  
**Dataset:** Our World in Data — COVID-19 Public Dataset  
**Goal:** Load, clean, explore, and visualize global COVID-19 trends to understand case growth, death rates, and vaccination rollout patterns.

Dataset source: https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

## 1. Data Loading

In [ ]:
url = 'https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv'
df = pd.read_csv(url, parse_dates=['date'])
print(f'Shape: {df.shape}')
df.head()

## 2. Data Cleaning

In [ ]:
# Check missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
print('Top 10 columns with most missing data:')
print(missing_report.sort_values('missing_pct', ascending=False).head(10))

In [ ]:
# Keep key columns and drop aggregate regions (OWID uses 'OWID_' prefix for non-countries)
key_cols = [
    'iso_code', 'continent', 'location', 'date',
    'total_cases', 'new_cases', 'total_deaths', 'new_deaths',
    'total_vaccinations', 'people_vaccinated', 'population',
    'total_cases_per_million', 'total_deaths_per_million'
]

df_clean = df[key_cols].copy()
df_clean = df_clean[~df_clean['iso_code'].str.startswith('OWID', na=True)]
df_clean = df_clean.dropna(subset=['continent'])

# Fill numeric NAs with 0 for cumulative counts
num_cols = ['total_cases', 'new_cases', 'total_deaths', 'new_deaths', 'total_vaccinations', 'people_vaccinated']
df_clean[num_cols] = df_clean[num_cols].fillna(0)

print(f'Cleaned shape: {df_clean.shape}')
print(f'Date range: {df_clean.date.min()} → {df_clean.date.max()}')
df_clean.dtypes

## 3. Descriptive Statistics

In [ ]:
# Latest snapshot per country
latest = df_clean.sort_values('date').groupby('location').last().reset_index()

print('=== Global Summary (latest data per country) ===')
print(f"Total countries: {latest['location'].nunique()}")
print(f"Total cases (global): {latest['total_cases'].sum():,.0f}")
print(f"Total deaths (global): {latest['total_deaths'].sum():,.0f}")
print(f"Global case fatality rate: {(latest['total_deaths'].sum() / latest['total_cases'].sum() * 100):.2f}%")
print()
print('=== Top 10 Countries by Total Cases ===')
print(latest.nlargest(10, 'total_cases')[['location', 'total_cases', 'total_deaths', 'people_vaccinated']].to_string(index=False))

In [ ]:
# Stats by continent
continent_stats = latest.groupby('continent').agg(
    total_cases=('total_cases', 'sum'),
    total_deaths=('total_deaths', 'sum'),
    vaccinated=('people_vaccinated', 'sum'),
    countries=('location', 'count')
).reset_index()
continent_stats['cfr'] = (continent_stats['total_deaths'] / continent_stats['total_cases'] * 100).round(2)
print(continent_stats.sort_values('total_cases', ascending=False).to_string(index=False))

## 4. Visualizations
### Viz 1 — Global Daily New Cases Over Time (7-day rolling average)

In [ ]:
global_daily = df_clean.groupby('date')['new_cases'].sum().reset_index()
global_daily['rolling_avg'] = global_daily['new_cases'].rolling(7).mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(global_daily['date'], global_daily['new_cases'], alpha=0.2, color='steelblue', label='Daily cases')
ax.plot(global_daily['date'], global_daily['rolling_avg'], color='steelblue', linewidth=2, label='7-day rolling avg')
ax.set_title('Global Daily New COVID-19 Cases with 7-Day Rolling Average', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('New Cases')
ax.legend()
plt.tight_layout()
plt.savefig('viz1_global_cases_over_time.png', dpi=150)
plt.show()
print('Finding: Clear wave structure visible — Omicron wave (early 2022) dwarfs all prior peaks in case count.')

### Viz 2 — Total Cases per Million by Continent (Box Plot)

In [ ]:
latest_filtered = latest[latest['total_cases_per_million'] > 0]

fig, ax = plt.subplots(figsize=(12, 5))
order = latest_filtered.groupby('continent')['total_cases_per_million'].median().sort_values(ascending=False).index
sns.boxplot(data=latest_filtered, x='continent', y='total_cases_per_million', order=order, ax=ax, palette='Set2')
ax.set_title('Distribution of Total COVID-19 Cases per Million by Continent', fontsize=14)
ax.set_xlabel('Continent')
ax.set_ylabel('Total Cases per Million')
plt.tight_layout()
plt.savefig('viz2_cases_per_million_continent.png', dpi=150)
plt.show()
print('Finding: Europe and South America show highest median cases per million, while Africa shows lowest — likely due to underreporting and younger demographics.')

### Viz 3 — Case Fatality Rate vs Vaccination Rate (Scatter, top 50 countries)

In [ ]:
scatter_df = latest[latest['total_cases'] > 100000].copy()
scatter_df['cfr'] = scatter_df['total_deaths'] / scatter_df['total_cases'] * 100
scatter_df['vax_rate'] = scatter_df['people_vaccinated'] / scatter_df['population'] * 100
scatter_df = scatter_df.dropna(subset=['cfr', 'vax_rate'])
scatter_df = scatter_df[scatter_df['vax_rate'] <= 100]

fig, ax = plt.subplots(figsize=(12, 6))
continents = scatter_df['continent'].unique()
colors = sns.color_palette('tab10', len(continents))
color_map = dict(zip(continents, colors))

for cont in continents:
    sub = scatter_df[scatter_df['continent'] == cont]
    ax.scatter(sub['vax_rate'], sub['cfr'], label=cont, color=color_map[cont], alpha=0.7, s=60)

# Trend line
z = np.polyfit(scatter_df['vax_rate'].dropna(), scatter_df['cfr'].dropna(), 1)
p = np.poly1d(z)
x_line = np.linspace(scatter_df['vax_rate'].min(), scatter_df['vax_rate'].max(), 100)
ax.plot(x_line, p(x_line), 'k--', linewidth=1.5, label='Trend')

ax.set_title('Case Fatality Rate vs Vaccination Rate by Country', fontsize=14)
ax.set_xlabel('% Population Vaccinated (at least 1 dose)')
ax.set_ylabel('Case Fatality Rate (%)')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig('viz3_cfr_vs_vaccination.png', dpi=150)
plt.show()
print('Finding: Negative trend — higher vaccination rates are associated with lower CFR, though causality is confounded by healthcare system quality.')

## 5. Key Findings Summary

1. **Wave structure**: COVID-19 spread in distinct waves globally. The Omicron variant (Jan 2022) produced the highest daily case counts by far, though deaths did not scale proportionally — suggesting reduced severity or better treatment.

2. **Geographic disparity**: Europe and South America recorded the highest cases per million. Africa's lower reported numbers likely reflect both genuine differences (age demographics) and data limitations (underreporting).

3. **Vaccination & mortality**: Countries with higher vaccination rates show a measurable reduction in case fatality rate. The relationship is not perfectly linear — healthcare infrastructure remains a significant confounding factor.

**Dataset**: Our World in Data COVID-19 dataset (https://ourworldindata.org/covid-deaths) — updated daily, covers 200+ countries from Jan 2020 onwards.